# Stage 4a — Choosing the aggregation architecture

**What this decides.** How the per-file piece embeddings are combined into one commit
representation, and how a piece longer than 512 tokens is split. Stage 4b then fine-tunes the
winner end to end.

**Why it is separate.** Comparing pooling strategies by fine-tuning each one would cost hours per
variant. Here the encoder is **frozen**: every piece is encoded once, the vectors are cached, and
only the small head on top is trained. Each comparison is then seconds, so the options get a fair
test instead of a rushed one.

**The caveat that comes with freezing.** The best pooling over frozen embeddings is not guaranteed
to stay best after the encoder adapts. The top two are carried into 4b rather than only the winner.

### The test set is not touched here

Stage 2's 173 test rows stay sealed. Comparing architectures on them and then reporting the
winner's score on the same rows would be selection bias — the number would reflect which variant
happened to suit those particular commits.

All comparisons below use a validation split carved from Stage 2's **training** rows. The test set
is opened once, in 4b.

## 0. Setup

Run this cell first. It works in Google Colab, Jupyter and IDEs such as VS Code or PyCharm.

It finds this repository (on Colab, opened straight from GitHub, it clones it), then picks one
working folder, `WORK`, where this notebook reads its inputs and writes its outputs. Every
notebook in the repository uses the same `WORK`, so they hand files to each other through it.

- set the environment variable `CIC_WORK` to choose the folder yourself
- otherwise it is `<repository>/work`, or on Colab a folder on your Google Drive
  (set `CIC_DRIVE=0` to stay on Colab's local disk)

An input missing from `WORK` is copied from the repository or downloaded from this study's Zenodo
record or Hugging Face repository, so any notebook can be run without re-running the ones before
it. See `replication.py`.

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO = "https://github.com/Binamra00/commit-intent-and-refactoring-levels"
_candidates = [Path(os.environ["CIC_ROOT"])] if os.environ.get("CIC_ROOT") else []
_candidates += [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in _candidates if (p / "replication.py").exists()), None)
if ROOT is None:                      # opened on its own, e.g. from GitHub in Colab
    ROOT = Path.cwd() / REPO.rsplit("/", 1)[-1]
    if not (ROOT / "replication.py").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO, str(ROOT)], check=True)
sys.path.insert(0, str(ROOT))

from replication import setup, require
WORK = setup()

## 1. Inputs

Stage 3a's diffs, Stage 2's split indices, and the representation Stage 3b settled on.

In [2]:
import json, re, math, time, collections
from pathlib import Path
import numpy as np
import pandas as pd

need = ["levin_with_diffs.jsonl", "stage2_train_idx.npy", "stage2_test_idx.npy",
        "levin_dataset.pkl"]
missing = [f for f in need if require(f, optional=True) is None]
if missing:
    raise SystemExit(f"missing {missing} -- run Stages 1-3 first")

D = pd.read_pickle("levin_dataset.pkl")
R = pd.DataFrame([json.loads(l) for l in open("levin_with_diffs.jsonl", encoding="utf-8")
                  if l.strip()])
R = R[R["ok"] == True].reset_index(drop=True)

# Stage 3b's exclusions, reapplied so the population matches.
R["java_files"] = R["files"].map(
    lambda fs: sum(1 for f in (fs or []) if str(f.get("filename", "")).endswith(".java")))
R = R[(R["truncated"] != True) & (R["java_files"] > 0)].reset_index(drop=True)

print(f"{len(R):,} commits")
print(R["label"].value_counts().to_string())

# Stage 2 split its own frame; align by commit id so the same commits land on the same side.
ID_COL = json.load(open("levin_columns.json"))["hash"]
d2 = D.drop_duplicates(subset=[ID_COL], keep="first").reset_index(drop=True)
tr_ids = set(d2.iloc[np.load("stage2_train_idx.npy")][ID_COL])
te_ids = set(d2.iloc[np.load("stage2_test_idx.npy")][ID_COL])

R["split"] = R["commit_id"].map(
    lambda c: "train" if c in tr_ids else ("test" if c in te_ids else "unassigned"))
print(f"\n{R['split'].value_counts().to_string()}")
if (R.split == "unassigned").any():
    print("\nCommits not in either Stage 2 split -- excluded from comparison")
    R = R[R.split != "unassigned"].reset_index(drop=True)

1,150 commits
label
corrective    500
perfective    404
adaptive      246

split
train    977
test     173


### Validation split

Carved from the training rows, stratified by label. The test rows are set aside and not referenced
again in this notebook.

In [3]:
from sklearn.model_selection import train_test_split

SEED = 42
train_pool = R.index[R.split == "train"].to_numpy()
TEST_IDX = R.index[R.split == "test"].to_numpy()

fit_idx, val_idx = train_test_split(
    train_pool, test_size=0.15, random_state=SEED,
    stratify=R.loc[train_pool, "label"])

print(f"fit        {len(fit_idx):,}")
print(f"validation {len(val_idx):,}   <- every comparison below is scored here")
print(f"test       {len(TEST_IDX):,}   <- sealed until Stage 4b")
print()
print(pd.DataFrame({
    "fit": R.loc[fit_idx, "label"].value_counts(),
    "validation": R.loc[val_idx, "label"].value_counts(),
}).to_string())

fit        830
validation 147   <- every comparison below is scored here
test       173   <- sealed until Stage 4b

            fit  validation
label                      
corrective  361          64
perfective  291          52
adaptive    178          31


## 2. Pieces

One piece for the message, one per changed file, in the structured form Stage 3b chose. A piece
longer than 512 tokens is split into windows — the stride is one of the things being decided, so
splitting happens at encode time rather than here.

In [4]:
HUNK = re.compile(r"^@@ .*? @@")

def message_of(rec):
    return (rec.get("git_message") or rec.get("dataset_comment") or "").strip()


def structured_pieces(rec):
    out = [f"<COMMIT_MESSAGE>{message_of(rec)}</COMMIT_MESSAGE>"]
    for f in rec.get("files") or []:
        added, removed = [], []
        for ln in (f.get("patch") or "").split("\n"):
            if HUNK.match(ln) or ln.startswith(("--- ", "+++ ")):
                continue
            if ln.startswith("+"):
                added.append(ln[1:])
            elif ln.startswith("-"):
                removed.append(ln[1:])
        piece = [f"<FILE>{f.get('filename')}</FILE>"]
        if removed:
            piece.append("<REMOVED>" + "\n".join(removed) + "</REMOVED>")
        if added:
            piece.append("<ADDED>" + "\n".join(added) + "</ADDED>")
        out.append("\n".join(piece))
    return out


PIECES = R.apply(structured_pieces, axis=1)
print(f"{int(PIECES.map(len).sum()):,} pieces across {len(R):,} commits")
print(f"median {int(PIECES.map(len).median())} per commit, max {int(PIECES.map(len).max())}")

7,181 pieces across 1,150 commits
median 3 per commit, max 248


## 3. Encode once, with the encoder frozen

Every window of every piece is passed through CodeBERT with gradients off, and the `[CLS]` vector
kept. This is the only expensive step, and it runs once for the whole notebook — the comparisons
afterwards reuse the cache.

**Stride is a parameter here.** A piece over 512 tokens is cut into windows; whether they overlap
is one of the decisions. No overlap is cheapest; half overlap gives the model a second view of
every boundary, at roughly double the passes for long pieces.

In [5]:
import torch
from transformers import AutoTokenizer, AutoModel

MODEL = "microsoft/codebert-base"
MAXLEN = 512
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {DEVICE}")

tok = AutoTokenizer.from_pretrained(MODEL)
enc = AutoModel.from_pretrained(MODEL).to(DEVICE).eval()
for p in enc.parameters():
    p.requires_grad = False
HID = enc.config.hidden_size
print(f"{MODEL}, hidden size {HID}, encoder frozen")


def windows(text, stride):
    """Token windows for one piece. Nothing is discarded; a long piece yields several windows."""
    ids = tok.encode(text, add_special_tokens=False, truncation=False)
    body = MAXLEN - 2                                   # room for <s> and </s>
    if len(ids) <= body:
        return [ids]
    step = body if stride is None else stride
    return [ids[i:i + body] for i in range(0, len(ids), step) if i < len(ids)]


@torch.no_grad()
def encode_all(stride, batch=32):
    """
    commit index -> array of piece embeddings.
    A piece that needed several windows is mean-pooled across its own windows first, so the
    commit-level pooling being compared always sees one vector per piece.
    """
    flat, owner = [], []
    for ci, ps in enumerate(PIECES):
        for pi, p in enumerate(ps):
            for w in windows(p, stride):
                flat.append(w); owner.append((ci, pi))

    vecs = np.zeros((len(flat), HID), dtype=np.float32)
    t0 = time.time()
    for s in range(0, len(flat), batch):
        chunk = flat[s:s + batch]
        L = max(len(c) for c in chunk) + 2
        ids = torch.full((len(chunk), L), tok.pad_token_id, dtype=torch.long)
        att = torch.zeros((len(chunk), L), dtype=torch.long)
        for i, c in enumerate(chunk):
            seq = [tok.cls_token_id] + c + [tok.sep_token_id]
            ids[i, :len(seq)] = torch.tensor(seq); att[i, :len(seq)] = 1
        out = enc(input_ids=ids.to(DEVICE), attention_mask=att.to(DEVICE))
        vecs[s:s + len(chunk)] = out.last_hidden_state[:, 0].cpu().numpy()
        if s % (batch * 50) == 0:
            done = s + len(chunk)
            print(f"\r   {done:,}/{len(flat):,} windows  "
                  f"{done/max(1e-9, time.time()-t0):.0f}/s", end="", flush=True)
    print(f"\r   {len(flat):,} windows in {(time.time()-t0)/60:.1f} min" + " " * 20)

    # windows -> pieces -> commits
    bypiece = collections.defaultdict(list)
    for v, (ci, pi) in zip(vecs, owner):
        bypiece[(ci, pi)].append(v)
    out = [[] for _ in range(len(PIECES))]
    for (ci, pi), vs in sorted(bypiece.items()):
        out[ci].append(np.mean(vs, axis=0))
    return [np.stack(v) for v in out]


STRIDES = {"no overlap": None, "half overlap": 256}
CACHE = {}
for name, s in STRIDES.items():
    print(f"\nstride: {name}")
    CACHE[name] = encode_all(s)

device: cuda


config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

... (3 outputs left out)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors: downloading bytes:           |  0.00B            

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (4884 > 512). Running this sequence through the model will result in indexing errors


microsoft/codebert-base, hidden size 768, encoder frozen

stride: no overlap
   12,081 windows in 6.4 min                    

stride: half overlap
   17,581 windows in 9.7 min                    


## 4. The pooling strategies

Four ways to turn a variable number of piece vectors into one commit vector.

| | how |
|---|---|
| **mean** | average of the pieces. The default, and the one most exposed to dilution — a commit with 248 pieces averages them all into one vector. |
| **max** | element-wise maximum. Keeps the strongest signal on each dimension regardless of how many pieces there are. |
| **attention** | a learned score per piece, then a weighted sum. Lets the model decide which files matter. |
| **transformer** | a single self-attention layer over the pieces, then mean. Most expressive, most parameters, and the most likely to overfit 1,151 examples. |

All four sit on the same frozen embeddings and are trained with the same optimiser and schedule, so
the comparison isolates the pooling.

In [6]:
import torch.nn as nn

LABELS = ["adaptive", "corrective", "perfective"]
L2I = {l: i for i, l in enumerate(LABELS)}
Y = R["label"].map(L2I).to_numpy()


class Pooler(nn.Module):
    def __init__(self, kind, hid=768, n_classes=3, dropout=0.2):
        super().__init__()
        self.kind = kind
        if kind == "attention":
            self.score = nn.Linear(hid, 1)
        if kind == "transformer":
            layer = nn.TransformerEncoderLayer(hid, nhead=4, dim_feedforward=512,
                                               dropout=dropout, batch_first=True)
            self.tr = nn.TransformerEncoder(layer, num_layers=1)
        self.drop = nn.Dropout(dropout)
        self.out = nn.Linear(hid, n_classes)

    def forward(self, x, mask):
        # x: (batch, pieces, hid);  mask: (batch, pieces) with 1 for a real piece
        m = mask.unsqueeze(-1)
        if self.kind == "mean":
            v = (x * m).sum(1) / m.sum(1).clamp(min=1)
        elif self.kind == "max":
            v = x.masked_fill(m == 0, -1e9).max(1).values
        elif self.kind == "attention":
            a = self.score(x).masked_fill(m == 0, -1e9).softmax(1)
            v = (x * a).sum(1)
        elif self.kind == "transformer":
            h = self.tr(x, src_key_padding_mask=(mask == 0))
            v = (h * m).sum(1) / m.sum(1).clamp(min=1)
        else:
            raise ValueError(self.kind)
        return self.out(self.drop(v))


def pad_batch(embs, idx, device):
    n = max(len(embs[i]) for i in idx)
    x = torch.zeros(len(idx), n, HID)
    m = torch.zeros(len(idx), n)
    for r, i in enumerate(idx):
        e = embs[i]
        x[r, :len(e)] = torch.from_numpy(e); m[r, :len(e)] = 1
    return x.to(device), m.to(device)


print(f"{len(LABELS)} classes: {LABELS}")
print(f"class counts: {dict(pd.Series(Y).value_counts().sort_index())}")

3 classes: ['adaptive', 'corrective', 'perfective']
class counts: {0: np.int64(246), 1: np.int64(500), 2: np.int64(404)}


### Class weights

The classes are uneven — corrective outnumbers adaptive two to one. Stage 2's tree ensembles
absorbed that implicitly; a neural network will not, and would under-predict adaptive without
help. The loss is weighted by inverse frequency, computed on the fitting rows only.

In [7]:
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score

counts = np.bincount(Y[fit_idx], minlength=len(LABELS))
weights = torch.tensor(counts.sum() / (len(LABELS) * counts), dtype=torch.float32)
print("inverse-frequency weights\n")
for l, c, w in zip(LABELS, counts, weights):
    print(f"   {l:<12}{c:>5} examples   weight {w:.3f}")


def run(kind, embs, fit, val, epochs=40, lr=1e-3, patience=6, seed=SEED, verbose=False):
    """Train one pooling head on frozen embeddings. Early stopping on validation kappa."""
    torch.manual_seed(seed); np.random.seed(seed)
    model = Pooler(kind, HID, len(LABELS)).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    lossf = nn.CrossEntropyLoss(weight=weights.to(DEVICE))

    yv = torch.tensor(Y[val]).to(DEVICE)
    best, best_state, bad = -1, None, 0
    for ep in range(epochs):
        model.train()
        order = np.random.permutation(fit)
        for s in range(0, len(order), 32):
            b = order[s:s + 32]
            x, m = pad_batch(embs, b, DEVICE)
            loss = lossf(model(x, m), torch.tensor(Y[b]).to(DEVICE))
            opt.zero_grad(); loss.backward(); opt.step()

        model.eval()
        with torch.no_grad():
            x, m = pad_batch(embs, val, DEVICE)
            pv = model(x, m).argmax(1).cpu().numpy()
        k = cohen_kappa_score(Y[val], pv)
        if k > best:
            best, best_state, bad = k, {n: p.detach().clone()
                                        for n, p in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= patience:
                break
        if verbose:
            print(f"      epoch {ep:>2}  kappa {k:.3f}")

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        x, m = pad_batch(embs, val, DEVICE)
        pv = model(x, m).argmax(1).cpu().numpy()
    return model, {
        "accuracy": round(accuracy_score(Y[val], pv) * 100, 1),
        "kappa": round(cohen_kappa_score(Y[val], pv) * 100, 1),
        "macro_f1": round(f1_score(Y[val], pv, average="macro") * 100, 1),
        "epochs": ep + 1,
    }

inverse-frequency weights

   adaptive      178 examples   weight 1.554
   corrective    361 examples   weight 0.766
   perfective    291 examples   weight 0.951


## 5. The comparison

Every pooling strategy against every stride. Each configuration is run with three seeds, because a
small head on 800-odd examples is noisy and a single run would rank on luck.

In [8]:
POOLINGS = ["mean", "max", "attention", "transformer"]
SEEDS = [42, 1337, 2024]

rows = []
for stride_name, embs in CACHE.items():
    for kind in POOLINGS:
        scores = []
        for sd in SEEDS:
            _, s = run(kind, embs, fit_idx, val_idx, seed=sd)
            scores.append(s)
        rows.append({
            "pooling": kind, "stride": stride_name,
            "kappa": round(np.mean([s["kappa"] for s in scores]), 1),
            "kappa sd": round(np.std([s["kappa"] for s in scores]), 1),
            "accuracy": round(np.mean([s["accuracy"] for s in scores]), 1),
            "macro f1": round(np.mean([s["macro_f1"] for s in scores]), 1),
            "epochs": int(np.mean([s["epochs"] for s in scores])),
        })
        print(f"   {stride_name:<14}{kind:<13}kappa {rows[-1]['kappa']:>5.1f} "
              f"(sd {rows[-1]['kappa sd']:.1f})")

A = pd.DataFrame(rows).sort_values("kappa", ascending=False)
print("\n\nVALIDATION RESULTS\n")
print(A.to_string(index=False))

   no overlap    mean         kappa  32.5 (sd 1.0)
   no overlap    max          kappa  36.2 (sd 2.3)
   no overlap    attention    kappa  40.4 (sd 3.0)


/usr/local/lib/python3.13/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


   no overlap    transformer  kappa  43.6 (sd 5.3)
   half overlap  mean         kappa  30.5 (sd 1.6)
   half overlap  max          kappa  38.2 (sd 1.0)
   half overlap  attention    kappa  38.8 (sd 3.3)
   half overlap  transformer  kappa  44.1 (sd 1.3)


VALIDATION RESULTS

    pooling       stride  kappa  kappa sd  accuracy  macro f1  epochs
transformer half overlap   44.1       1.3      63.3      60.9      29
transformer   no overlap   43.6       5.3      64.4      61.6      25
  attention   no overlap   40.4       3.0      61.7      60.0      23
  attention half overlap   38.8       3.3      60.3      58.7      26
        max half overlap   38.2       1.0      59.2      58.6      25
        max   no overlap   36.2       2.3      58.5      56.7      19
       mean   no overlap   32.5       1.0      55.8      55.1      23
       mean half overlap   30.5       1.6      54.7      53.0      18


### Does the spread across seeds swamp the differences?

If the gap between the best and second-best is smaller than the run-to-run variation, the ranking
is noise and both should go forward.

In [9]:
top = A.iloc[0]
second = A.iloc[1]
gap = top["kappa"] - second["kappa"]
noise = max(top["kappa sd"], second["kappa sd"])

print(f"best        : {top['pooling']} / {top['stride']}   kappa {top['kappa']:.1f}")
print(f"second      : {second['pooling']} / {second['stride']}   kappa {second['kappa']:.1f}")
print(f"gap         : {gap:.1f}")
print(f"seed spread : {noise:.1f}")
print()
if gap <= noise:
    print("The gap is within run-to-run variation. The ranking is not reliable -- carry both into")
    print("Stage 4b and let end-to-end fine-tuning separate them.")
else:
    print("The gap exceeds the seed spread, so the ranking is meaningful. The second is still")
    print("carried forward, because the best pooling over frozen embeddings need not stay best")
    print("once the encoder adapts.")

print("\n\nDOES STRIDE MATTER?\n")
st = A.pivot_table(index="pooling", columns="stride", values="kappa")
st["difference"] = (st.get("half overlap", 0) - st.get("no overlap", 0)).round(1)
print(st.to_string())
print("""
Overlap only affects pieces longer than 512 tokens, which are a minority. If the difference is
small, take no overlap -- it is cheaper and simpler to describe.""")

best        : transformer / half overlap   kappa 44.1
second      : transformer / no overlap   kappa 43.6
gap         : 0.5
seed spread : 5.3

The gap is within run-to-run variation. The ranking is not reliable -- carry both into
Stage 4b and let end-to-end fine-tuning separate them.


DOES STRIDE MATTER?

stride       half overlap  no overlap  difference
pooling                                          
attention            38.8        40.4        -1.6
max                  38.2        36.2         2.0
mean                 30.5        32.5        -2.0
transformer          44.1        43.6         0.5

Overlap only affects pieces longer than 512 tokens, which are a minority. If the difference is
small, take no overlap -- it is cheaper and simpler to describe.


## 6. A floor to beat

Frozen CodeBERT embeddings with a trained head should beat a bag-of-words baseline on the same
rows. If it does not, the embeddings are contributing nothing and fine-tuning in 4b is unlikely to
rescue it.

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

texts = R.apply(lambda r: "\n".join(structured_pieces(r)), axis=1).to_numpy()
vec = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), sublinear_tf=True)
Xtr = vec.fit_transform(texts[fit_idx])
Xva = vec.transform(texts[val_idx])
lr = LogisticRegression(max_iter=2000, class_weight="balanced").fit(Xtr, Y[fit_idx])
pv = lr.predict(Xva)

base_k = cohen_kappa_score(Y[val_idx], pv) * 100
base_a = accuracy_score(Y[val_idx], pv) * 100
print(f"TF-IDF + logistic regression : accuracy {base_a:.1f}%   kappa {base_k:.1f}")
print(f"best frozen-CodeBERT head    : accuracy {top['accuracy']:.1f}%   kappa {top['kappa']:.1f}")
print()
if top["kappa"] > base_k + 2:
    print("The embeddings add signal over surface word counts. Proceed to fine-tuning.")
else:
    print("Frozen embeddings are not beating bag-of-words. That is worth understanding before")
    print("spending 4b on fine-tuning -- check the encoding, the pooling, and whether the")
    print("structured markers are being tokenised sensibly.")

TF-IDF + logistic regression : accuracy 58.5%   kappa 36.0
best frozen-CodeBERT head    : accuracy 63.3%   kappa 44.1

The embeddings add signal over surface word counts. Proceed to fine-tuning.


## 7. Carry forward

In [11]:
carry = A.head(2)[["pooling", "stride", "kappa", "kappa sd"]].to_dict("records")
decision = {
    "validation_rows": int(len(val_idx)),
    "fit_rows": int(len(fit_idx)),
    "test_rows_sealed": int(len(TEST_IDX)),
    "seeds": SEEDS,
    "results": A.to_dict("records"),
    "carry_to_4b": carry,
    "tfidf_baseline": {"accuracy": round(base_a, 1), "kappa": round(base_k, 1)},
    "class_weights": {l: round(float(w), 3) for l, w in zip(LABELS, weights)},
}
json.dump(decision, open("stage4a_decision.json", "w"), indent=1)
A.to_csv("stage4a_results.csv", index=False)

print("=" * 72)
print("STAGE 4A DECISION")
print("=" * 72)
for c in carry:
    print(f"   {c['pooling']:<13}{c['stride']:<15}kappa {c['kappa']:.1f} (sd {c['kappa sd']:.1f})")
print(f"""
Both go to Stage 4b, where the encoder is unfrozen. Frozen-embedding rankings do not always
survive fine-tuning, and carrying two costs one extra training run.

Stage 4b must:
   - fine-tune end to end with AdamW at 2e-5 to 5e-5, not the 1e-3 used here for a small head
   - keep the class weights above
   - early-stop on validation, 3 to 4 epochs is usually enough
   - evaluate on Stage 2's test indices ONCE, at the end

   stage4a_decision.json and stage4a_results.csv written""")

STAGE 4A DECISION
   transformer  half overlap   kappa 44.1 (sd 1.3)
   transformer  no overlap     kappa 43.6 (sd 5.3)

Both go to Stage 4b, where the encoder is unfrozen. Frozen-embedding rankings do not always
survive fine-tuning, and carrying two costs one extra training run.

Stage 4b must:
   - fine-tune end to end with AdamW at 2e-5 to 5e-5, not the 1e-3 used here for a small head
   - keep the class weights above
   - early-stop on validation, 3 to 4 epochs is usually enough
   - evaluate on Stage 2's test indices ONCE, at the end

   stage4a_decision.json and stage4a_results.csv written
